# 3. Network extraction

This notebook walks through the steps of:
1. Extract networks from text
2. Calculating network statistics

Note that hyperparamaters used for UMAP and hdbscan are determined through through process in `2_HyperparameterTuning.ipynb`.

Input:\
`data/ANES_2016_CLEANED.txt` : a tab-seperated file with `_clean` fields for every text response. 

Output:\
`data/network_data.csv` : a file with extracted networks (node + edge lists) and network statistics.

In [1]:
# helpful packages
import pandas as pd
import numpy as np
import itertools as it

# language models
import spacy
from spacy.tokens import Doc
nlp = spacy.load('en_core_web_trf')

# language clustering
from gensim.models import KeyedVectors
import umap
import hdbscan

# network measures
import networkx as nx
import netstats as ns

import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Load cleaned data

In [2]:
# read cleaned data
df = pd.read_csv('data/ANES_2016_CLEANED.txt', sep='\t')

print(f'Data includes {len(df)} observations on {len(df.columns)} variables.')
print(f'\nRecorded variables are:\n{", ".join(df.columns)}')

Data includes 4270 observations on 30 variables.

Recorded variables are:
V160001, V160001_orig, age, female, education, leftright, politicalinterest, politicalparticipation, TIPI_extraversion, TIPI_agreeableness, TIPI_conscientiousness, TIPI_emotionalstability, TIPI_openness, V161069, V161072, V161075, V161078, V161098, V161101, V161104, V161106, party_id, V161069_clean, V161072_clean, V161075_clean, V161078_clean, V161098_clean, V161101_clean, V161104_clean, V161106_clean


In [3]:
# Questions with free-response answers. 
text_qs = ['V161069', # PRE: Text- What is it that R likes about Democratic Pres cand
           'V161072', # PRE: Text- What is it that R dislikes about Democratic Pres cand
           'V161075', # PRE: Text- What is it that R likes about Republican Pres cand
           'V161078', # PRE: Text- What is it that R dislikes about Republican Pres cand
           'V161098', # PRE: Text- What does R like about Democratic party
           'V161101', # PRE: Text- What does R dislike about the Democratic party
           'V161104', # PRE: Text- What does R like about Republican party
           'V161106'  # PRE: Text- What does R dislike about the Republican party
          ]

In [4]:
# create spacy docs of each response
for q in text_qs:
    print(f'  Processing question {q}')
    df[f'{q}_doc'] = df[f'{q}_clean'].fillna('').apply(lambda x: nlp(x))
    
print('All text converted to SpaCy.')

  Processing question V161069
  Processing question V161072
  Processing question V161075
  Processing question V161078
  Processing question V161098
  Processing question V161101
  Processing question V161106
All text converted to SpaCy.


# 3. Network extraction

### Step 1: Retained Nodes & Lemmatization

In [5]:
# keep as nodes word that are these parts of speech:
keep_pos = ['ADJ', 'ADV', 'NOUN', 'PROPN', 'VERB']


raw_nodes = list()

for q in text_qs:
    docs = df[f'{q}_doc']

    for doc in docs:
        # nodes to keep (unclustered)
        for token in doc:
            if token.pos_ in keep_pos and token.lemma_ not in raw_nodes:
                raw_nodes.append(token.lemma_)
                
print(f'{len(raw_nodes)} words initially retained as nodes across all responses')

8760 words initially retained as nodes across all responses


### Step 2: Numeric Representation of Words (Nodes) Using Embeddings

In [6]:
# embeddings model
model = KeyedVectors.load_word2vec_format('../GoogleNews-vectors-negative300.bin', binary=True)


vecs = list() # embeddings for in-vocab words
kept = list() # words that have an embedding in vecs 
skipped = list() # out of vocab (dropped from analysis)

for word in raw_nodes:
    try:
        vector = model[word]
        vecs.append(vector) # save vector
        kept.append(word) # save word
    except:
        skipped.append(word) # out of vocab (after spell check correction)
    
word_embeddings = np.vstack(vecs)

w, n = word_embeddings.shape
print(f'{w} words embedded into {n}-dimensions.')
print(f'{len(skipped)} words dropped as out of vocabulary.')

6014 words embedded into 300-dimensions.
2746 words dropped as out of vocabulary.


### Step 3: Cluster word embeddings using UMap + hdbscan

Note: The hyperparameters used here were determined through an interative process until returned clusters were qualitatively meaningful. Eg, the words assigned to each cluster were determined through manual analysis to reasonably reflect the same concept. If working with a different corpus, hyperparamters should be tuned appropriately.

In [7]:
def run_clustering(n_components, min_cluster_size, min_samples, word_embeddings):
    
    # reduce dimensionality
    reducer = umap.UMAP(n_components = n_components, random_state=42, n_jobs=1) 
    embedding = reducer.fit_transform(word_embeddings)

    # cluster
    clusterer = hdbscan.HDBSCAN(min_cluster_size = min_cluster_size, 
                                min_samples = min_samples, 
                                prediction_data = True,
                                approx_min_span_tree=False)

    cluster_labels = clusterer.fit_predict(embedding)
    
    return cluster_labels

In [8]:
# hyperparameters
n_components = 25 
min_cluster_size = 6
min_samples = 7 

cluster_labels = run_clustering(n_components, min_cluster_size, min_samples, word_embeddings)    
        
# count number of clusters. Note: words assigned to "-1" (no cluster) retained as unique words
n_clusters = list(cluster_labels).count(-1) + max(cluster_labels) + 1 # 0-indexed

print(f'{n_clusters} found') #3981 found


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


3981 found


In [9]:
#### Save clusters as dictionaries
# Note: word lists should be manually inspected for coherence

# Dict 1: {num : [words]} 
clust_words = dict() 

for word, label in zip(kept, cluster_labels):
    clust_words.setdefault(label, set())
    clust_words[label].add(word)
    
    
# Dict 2:  {word : label}
word_label = dict() 

for word, clust in zip(kept, cluster_labels):

    if clust >= 0: # keep clustered words
        word_label[word] = list(clust_words[clust])[0]

### Step 4: Construct Network for Each Response

In [10]:
# flag negation

def check_negs(nodes, all_edges, neg_parents): 
    neg = nlp('NEG')[0]

    # add 'NEG' as node
    nodes.append(neg) 
        
    # add 'NEG' to edge list
    for p in neg_parents:
        
        # add 'NEG' as key
        all_edges[neg] = p.head
        del all_edges[p] # remove head
        
        # add 'NEG' as value
        for k, v in all_edges.items():
            if v == p:
                all_edges[k] = neg
                
    return nodes, all_edges

In [11]:
# identify child-parent relationships

def prune_edges(all_edges, nodes):
    edges = list()

    for child, parent in all_edges.items():
        if child in nodes: # if we have a child that is a node
            seen = set()

            while parent not in nodes and parent not in seen: # find a parent
                grand = all_edges[parent]
                seen.add(parent)

                if grand == parent: # if we're in a loop
                    break
                else:
                    parent = grand # iteratively check grand parents

            if child.text != parent.text and parent in nodes:
                edges.append([child, parent])
                
    return edges

In [12]:
def merge_nodes(nodes, edges):
                
    ##### NODES #####
    lemma_nodes = [node.lemma_ for node in nodes]
    final_nodes = set()
    has_neg = False
    
    for node in lemma_nodes:
        # case 1: replace with cluster label
        if node in word_label:
            final_nodes.add(word_label[node])
        # case 2: this is a node indicating negation (drop)
        elif node == 'NEG':
            has_neg = True
        # case 3: this is a node w/ no cluster label (add as is)
        else:
            final_nodes.add(node)
            
    #### EDGES #####
    # step 1: lemmatize & cluster
    clean_edges = list() # allows for duplicated edges
    
    for e1, e2 in edges:
        clean_edge = list()
        
        for e in [e1, e2]:
            lemma = e.lemma_

            if lemma in word_label: # if in cluster
                clean_edge.append(word_label[lemma])
            else:
                clean_edge.append(lemma)
                
        clean_edges.append(clean_edge)    
    
    keep_edges = list() # edges to add/keep

    neg_edge = list() # track pairs

    for e1, e2 in clean_edges:
        keep = True

        if e1=='NEG':
            neg_edge.append(e2)
            keep = False
        elif e2=='NEG':
            neg_edge.append(e1)
            keep = False

        # keep edges with no negation
        if keep:
            keep_edges.append([e1, e2, 1]) # positive weight


        # if we've found two things connected to negatives
        # taking advantage of edges being in order
        if len(neg_edge) == 2:
            neg_edge.append(-1) # negative weight
            keep_edges.append(neg_edge) # add to list of keep edges
            neg_edge = list() # clear
                
        # replace orig edges
        final_edges = keep_edges
        
    return final_nodes, final_edges

In [13]:
def get_network(doc):
    
    all_nodes = set()
    all_edges = list()
    skipped = 0

    for sent in doc.sents:
        try:
            # Step 1: Nodes (words to keep)
            nodes = [token for token in sent if token.pos_ in keep_pos]

            # Step 2: Edges (parse tree)
            ### 2a: all edges
            raw_edges = dict() # child : parent

            for token in sent:
                raw_edges[token] = token.head

            ### 2b: Check negatives
            neg_parents = [token.head for token in sent if token.dep_ == 'neg']

            if len(neg_parents) > 0:
                nodes, raw_edges = check_negs(nodes, raw_edges, neg_parents)

            # 2c: edges to keep
            edges = prune_edges(raw_edges, nodes)

            # Step 3: finalize based on word clusters, negatives
            final_nodes, final_edges = merge_nodes(nodes, edges)
        except:
            final_nodes = list()
            final_edges = list()
            skipped += 1
    
        # update for full doc
        for node in final_nodes:
            all_nodes.add(node)

        for e in final_edges:
            all_edges.append(e)
        
    network = {'nodes' : all_nodes,
               'edges' : all_edges}
    
    return network

## Network extraction

In [14]:
for q in text_qs:
    print(f'Processing question {q}')
    
    # get network from response text
    df[f'{q}_network'] = df[f'{q}_doc'].apply(lambda doc: get_network(doc))
    
print('All text converted to networks')

Processing question V161069
Processing question V161072
Processing question V161075
Processing question V161078
Processing question V161098
Processing question V161101
Processing question V161104
Processing question V161106
All text converted to networks


# Calculate Network Statistics

In [15]:
def get_network_stats(network):
    nodes = network['nodes']
    edges = network['edges']
    
    if len(nodes) > 1 and len(edges) > 1:
        G = nx.Graph()
        G.add_weighted_edges_from(edges)

        for node in nodes:
            G.add_node(node)

        stats = ns.network_stats(G)
    
    else:
        stats = dict()
        
    return stats

In [18]:
for q in text_qs:
    print(f'Processing question {q}')
    
    # get stats from network
    df[f'{q}_stats'] =  df[f'{q}_network'].apply(lambda network: get_network_stats(network))
    
print('All network statistic calculated.')

Processing question V161069
Processing question V161072
Processing question V161075
Processing question V161078
Processing question V161098
Processing question V161101
Processing question V161104
Processing question V161106
All network statistic calculated.


# Save to file for analysis

In [19]:
df.to_csv('data/network_data.csv', index=False)

# Check merges across all data

In [40]:
import ast

In [29]:
# Check how many nodes merged
def get_merge_count(doc):
    merges = 0

    for sent in doc.sents:
        # Step 1: Get raw nodes (words to keep)
        nodes = [token for token in sent if token.pos_ in keep_pos]
        lemma_nodes = [node.lemma_ for node in nodes]
        
        for i, j in it.combinations(lemma_nodes, 2):
            try:
                l1 = word_label[i]
                l2 = word_label[j]

                if l1 == l2: # if two words have the same cluster label that is one merge
                    merges += 1
            except KeyError:
                pass
    
    return merges

In [36]:
for q in text_qs:
    print(f'Processing question {q}')

    # count words merged in response text
    df[f'{q}_merges'] = df[f'{q}_doc'].apply(lambda doc: get_merge_count(doc))

Processing question V161069
Processing question V161072
Processing question V161075
Processing question V161078
Processing question V161098
Processing question V161101
Processing question V161104
Processing question V161106


In [37]:
count = 0

for q in text_qs:
    total_merges = np.sum(df[f'{q}_merges'])
    print(f' {total_merges} for Q {q}')
    
    count += total_merges
    
print(f'{count} merges across full data')

 329 for Q V161069
 270 for Q V161072
 365 for Q V161075
 416 for Q V161078
 373 for Q V161098
 400 for Q V161101
 270 for Q V161104
 362 for Q V161106
2785 merges across full data


In [41]:
# helper function to extract saved stats as dictionary
def to_dict(x):
    try:
        y = ast.literal_eval(x)
        if isinstance(y, dict):  # Ensure it's actually a dictionary
            return y
    except(ValueError, SyntaxError):
        return None  # Or handle the error as needed
    
net_cols = [f'{q}_network' for q in text_qs] # node/edge lists for question
stat_col = [f'{q}_stats' for q in text_qs] # network stats for each question

data = pd.read_csv('data/network_data.csv', 
                 converters=dict((col, to_dict) for col in stat_col + net_cols))


In [47]:
count = 0

for q in text_qs:
    stats = data[f'{q}_stats']
    nodes = np.sum([s['node_count'] for s in stats if 'node_count' in s])
     
    print(f' {nodes} nodes for Q {q}')
    
    count += nodes
    
print(f'{count} nodes across full data')

 7502 nodes for Q V161069
 9326 nodes for Q V161072
 7513 nodes for Q V161075
 11579 nodes for Q V161078
 8626 nodes for Q V161098
 9730 nodes for Q V161101
 7167 nodes for Q V161104
 10525 nodes for Q V161106
71968 nodes across full data
